# Count every word in a big text with MapReduce

Map on chunks with private dictionaries, shuffle by key, reduce by adding. The
text never moves; only the small (word, count) pairs do.

In [1]:
import re
from collections import Counter, defaultdict

text = ("the vote on the budget was close and the party split on the vote " * 2000).split()
chunks = [text[i:i + 4000] for i in range(0, len(text), 4000)]
print(len(text), "words in", len(chunks), "chunks")

28000 words in 7 chunks


In [2]:
def map_chunk(chunk):
    """Runs on one machine, on its own chunk, with its own dictionary. No shared state."""
    return Counter(chunk)

def shuffle(mapped):
    """Route every (word, count) pair to the reducer responsible for that word."""
    by_word = defaultdict(list)
    for counts in mapped:
        for word, n in counts.items():
            by_word[word].append(n)
    return by_word

def reduce_word(word, counts):
    return word, sum(counts)

mapped = [map_chunk(c) for c in chunks]               # in real MapReduce: in parallel
shuffled = shuffle(mapped)
result = dict(reduce_word(w, c) for w, c in shuffled.items())
print(sorted(result.items(), key=lambda kv: -kv[1])[:5])
print("same as counting directly:", result == Counter(text))

[('the', 8000), ('vote', 4000), ('on', 4000), ('budget', 2000), ('was', 2000)]
same as counting directly: True


## How much crossed the "network"?

In [3]:
pairs_sent = sum(len(c) for c in mapped)
print(f"{len(text):,} words of input;  {pairs_sent} (word, count) pairs sent to reducers")

28,000 words of input;  63 (word, count) pairs sent to reducers


## Run the map phase in parallel for real

In [4]:
from multiprocessing import Pool
try:
    with Pool(4) as pool:
        mapped_parallel = pool.map(map_chunk, chunks)
    print("parallel map gives the same result:", dict(reduce_word(w, c) for w, c in shuffle(mapped_parallel).items()) == result)
except Exception as e:
    print("multiprocessing not available here:", e)

parallel map gives the same result: True


Why it is not 4× faster: the shuffle and the reduce are serial, and the slowest
worker sets the finish line. This is Amdahl's law with a communication term.

## What does *not* distribute this way

Addition is associative and commutative, so partial counts combine in any order.
A median is not:

In [5]:
import statistics
chunk_a, chunk_b = [1, 2, 3, 4, 5, 6, 7, 8, 9], [10, 11]        # two machines, unequal chunks
print("median of medians:", statistics.median([statistics.median(chunk_a), statistics.median(chunk_b)]),
      "  true median:", statistics.median(chunk_a + chunk_b))
print("mean of means (weighted): ", (sum(chunk_a) + sum(chunk_b)) / (len(chunk_a) + len(chunk_b)), "  sums and counts combine exactly")

median of medians: 7.75   true median: 6
mean of means (weighted):  6.0   sums and counts combine exactly


**What we used:** MapReduce, which statistics split exactly, and Amdahl's law
(session 12, sections 9 to 11).